# Shard Large Graph Artifact (Colab)

This notebook converts a packed graph artifact such as `data/processed/graphs_master_ff_rich.pt` into a sharded directory artifact that loads lazily during training and benchmarking.

Workflow:
1. Mount Drive and locate the repo.
2. Run `scripts/shard_graph_artifact.py` directly against the Drive-mounted artifact.
3. Verify the finished `.sharded` artifact in the repo.

Notes:
- This job is CPU and disk bound, not GPU bound.
- The sharder prints a progress bar and ETA during shard writing.
- Training and benchmark scripts automatically prefer `<graphs>.pt.sharded` when it exists.


## 1) Mount Drive And Open Repo

In Colab, use a high-RAM runtime. GPU is optional for this notebook.


In [1]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)


def _resolve_repo_root() -> Path:
    env_repo = os.environ.get("FRM_REPO_DIR", "").strip()
    if env_repo:
        p = Path(env_repo)
        if (p / "configs" / "default.toml").exists():
            return p

    candidates = [
        Path.cwd(),
        Path("/content/Forward-Risk-Manager"),
        Path("/content/drive/MyDrive/Forward-Risk-Manager"),
        Path("/content/drive/MyDrive/forward-risk-manager"),
    ]
    if IN_COLAB:
        drive_root = Path("/content/drive/MyDrive")
        if drive_root.exists():
            candidates.extend(
                sorted(p for p in drive_root.glob("*Forward*Risk*Manager*") if p.is_dir())
            )

    for p in candidates:
        if (p / "configs" / "default.toml").exists():
            return p

    raise FileNotFoundError(
        "Could not find repo root containing configs/default.toml. "
        "Set FRM_REPO_DIR or update candidate paths in this cell."
    )


ROOT = _resolve_repo_root()
os.chdir(ROOT)
print("repo root:", ROOT)
print("cwd:", Path.cwd())


Mounted at /content/drive
repo root: /content/drive/MyDrive/Forward-Risk-Manager
cwd: /content/drive/MyDrive/Forward-Risk-Manager


## 2) Install Or Validate Minimal Dependencies


In [2]:
import importlib.util
import sys
from pathlib import Path

if str((ROOT / "src").resolve()) not in sys.path:
    sys.path.insert(0, str((ROOT / "src").resolve()))

from frisk.notebook_runtime import run_command, shell_quote

PYTHON_EXE = shell_quote(sys.executable)


def run(cmd: str, allow_fail: bool = False, tail_lines: int = 200) -> bool:
    result = run_command(
        cmd,
        allow_fail=allow_fail,
        tail_lines=tail_lines,
        log_dir=ROOT / "runs" / "experiments" / "_shard_graph_logs",
    )
    return result.ok


required_modules = ["torch", "torch_geometric", "tqdm"]
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]

if missing:
    print("Missing modules detected:", missing)
    run(f"{PYTHON_EXE} -m pip install --upgrade pip setuptools wheel")
    base_need = [m for m in missing if m != "torch_geometric"]
    if base_need:
        run(f"{PYTHON_EXE} -m pip install " + " ".join(shell_quote(m) for m in base_need))
    if "torch_geometric" in missing:
        run(f"{PYTHON_EXE} -m pip install torch-geometric")
else:
    print("Dependencies already available. Skipping install.")


Missing modules detected: ['torch_geometric']

/usr/bin/python3 -m pip install --upgrade pip setuptools wheel
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.9 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.

completed in 11.03s | log: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/_shard_graph_logs/1773617498_usr_bin_python3_-m_pip_install_--upgrade_pip_setuptools_wheel.log

/u

## 3) Sharding Configuration

This notebook shards the packed artifact directly inside the mounted repo on Drive.


In [3]:
import os

SOURCE_GRAPH = ROOT / "data" / "processed" / "graphs_master_ff_rich.pt"
SHARDED_GRAPH = ROOT / "data" / "processed" / "graphs_master_ff_rich.pt.sharded"

SHARD_SIZE = 256
TORCH_NUM_THREADS = max(1, os.cpu_count() or 1)
FORCE_REBUILD_SHARDS = False

assert SOURCE_GRAPH.exists(), f"Missing source graph artifact: {SOURCE_GRAPH}"

print("source:", SOURCE_GRAPH)
print("source_size_gb:", round(SOURCE_GRAPH.stat().st_size / (1024 ** 3), 2))
print("sharded_graph:", SHARDED_GRAPH)
print("torch_num_threads:", TORCH_NUM_THREADS)
print("shard_size:", SHARD_SIZE)


source: /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt
source_size_gb: 19.19
sharded_graph: /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt.sharded
torch_num_threads: 8
shard_size: 256


## 4) Shard Graph Artifact


In [4]:
import json
import shutil

if FORCE_REBUILD_SHARDS and SHARDED_GRAPH.exists():
    print("Removing existing sharded artifact before rebuild:", SHARDED_GRAPH)
    shutil.rmtree(SHARDED_GRAPH)

if not SHARDED_GRAPH.exists():
    ok = run(
        f"{PYTHON_EXE} -u scripts/shard_graph_artifact.py "
        f"--graphs {shell_quote(str(SOURCE_GRAPH))} "
        f"--out {shell_quote(str(SHARDED_GRAPH))} "
        f"--shard-size {SHARD_SIZE} "
        f"--torch-num-threads {TORCH_NUM_THREADS}",
        tail_lines=400,
    )
    assert ok, "Sharding command failed. See streamed logs above."
else:
    print("Reusing existing sharded artifact:", SHARDED_GRAPH)

manifest = json.loads((SHARDED_GRAPH / "manifest.json").read_text())
print("manifest_num_graphs:", manifest.get("num_graphs"))
print("manifest_num_shards:", manifest.get("num_shards"))
print("manifest_shard_size:", manifest.get("shard_size"))



/usr/bin/python3 -u scripts/shard_graph_artifact.py --graphs /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt --out /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt.sharded --shard-size 256 --torch-num-threads 8
torch_num_threads=8
source=/content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt size_gb=19.19
Loading packed graph artifact into CPU memory...
loaded format=packed graphs=10400 in 4.25 min
Writing sharded artifact to /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt.sharded (shard_size=256)...

Writing shards: 100%|██████████| 41/41 [02:21<00:00,  3.45s/shard]
Wrote sharded artifact: /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt.sharded
timing: load=4.25 min | write=2.36 min | total=6.61 min

completed in 403.63s | log: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/_shard_graph_logs/177361751

## 5) Artifact Check


In [5]:
from pathlib import Path
import json

manifest_path = SHARDED_GRAPH / "manifest.json"
assert SHARDED_GRAPH.exists(), f"Missing sharded artifact directory: {SHARDED_GRAPH}"
assert manifest_path.exists(), f"Missing manifest: {manifest_path}"

manifest = json.loads(manifest_path.read_text())
graph_shards = sorted((SHARDED_GRAPH / "shards").glob("*.pt"))
meta_shards = sorted((SHARDED_GRAPH / "meta").glob("*.pt"))

print("artifact_dir:", SHARDED_GRAPH)
print("artifact_size_gb:", round(sum(p.stat().st_size for p in SHARDED_GRAPH.rglob("*") if p.is_file()) / (1024 ** 3), 2))
print("num_graph_shards:", len(graph_shards))
print("num_meta_shards:", len(meta_shards))
print("manifest_num_graphs:", manifest.get("num_graphs"))
print("manifest_num_shards:", manifest.get("num_shards"))
print("manifest_keys:", sorted(manifest.keys()))

print("\nNext step:")
print("Run notebooks/paper_final_benchmark_colab.ipynb or notebooks/end_to_end_repo_runbook.ipynb.")
print("The training and benchmark scripts will automatically prefer this sharded artifact.")


artifact_dir: /content/drive/MyDrive/Forward-Risk-Manager/data/processed/graphs_master_ff_rich.pt.sharded
artifact_size_gb: 19.19
num_graph_shards: 41
num_meta_shards: 41
manifest_num_graphs: 10400
manifest_num_shards: 41
manifest_keys: ['config', 'dates', 'format', 'num_graphs', 'num_shards', 'shard_size', 'shards', 'stats', 'version']

Next step:
Run notebooks/paper_final_benchmark_colab.ipynb or notebooks/end_to_end_repo_runbook.ipynb.
The training and benchmark scripts will automatically prefer this sharded artifact.
